خروجی‌های ذخیره‌شده مربوط به آزمون کد روی **۵۰٬۰۰۰ ردیف اول دیتاست واقعی** هستند، نه کل یک میلیون آگهی.
ورودی همچنان `../data/Divar.csv` است؛ با اجرای دوباره سلول‌ها، نتایج فایل خودت جایگزین می‌شوند.
[منبع داده](https://huggingface.co/datasets/divarofficial/real_estate_ads)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
data_set = pd.read_csv("../data/Divar.csv")

C:\Users\sepeh\AppData\Local\Temp\ipykernel_11116\2814678356.py:1: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  data_set = pd.read_csv("../data/Divar.csv")


In [3]:
data_set["cat2_slug"].value_counts(dropna=False)

cat2_slug
residential-sell        558708
residential-rent        276558
commercial-rent          76567
commercial-sell          38861
temporary-rent           29903
real-estate-services     19403
Name: count, dtype: int64

In [4]:
luxury_features = [
    "has_pool",
    "has_barbecue",
    "has_sauna",
    "has_jacuzzi"
]

non_luxury_features = [
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_water",
    "has_warm_water_provider",
    "has_electricity",
    "has_gas",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "has_security_guard"
]

all_features = luxury_features + non_luxury_features
# has_business_deed noe sanade, emkanat nist

In [5]:
property_categories = [
    "residential-sell",
    "residential-rent",
    "commercial-sell",
    "commercial-rent"
]

housing_data = data_set.loc[
    data_set["cat2_slug"].isin(property_categories),
    ["cat2_slug", "cat3_slug", "price_mode", "price_value",
     "credit_mode", "rent_mode", "transformable_credit", "transformable_rent"]
    + all_features
].copy()

housing_data.head()

,cat2_slug,cat3_slug,price_mode,price_value,credit_mode,rent_mode,transformable_credit,transformable_rent,has_pool,has_barbecue,...,has_warehouse,has_parking,has_water,has_warm_water_provider,has_electricity,has_gas,has_heating_system,has_cooling_system,has_restroom,has_security_guard
1,residential-sell,apartment-sell,مقطوع,8.500000e+09,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,residential-rent,apartment-rent,NaN,NaN,مقطوع,مقطوع,750000000.0,26000000.0,NaN,NaN,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,commercial-rent,office-rent,NaN,NaN,مقطوع,مقطوع,950000000.0,95000000.0,NaN,NaN,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,residential-sell,apartment-sell,مقطوع,5.750000e+09,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,NaN,package,NaN,NaN,shoofaj,air_conditioner,squat_seat,NaN
5,residential-rent,apartment-rent,NaN,NaN,مقطوع,مقطوع,250000000.0,6000000.0,NaN,NaN,...,True,True,NaN,water_heater,NaN,NaN,NaN,NaN,squat,NaN


In [6]:
for feature in all_features:
    print(feature)
    print(housing_data[feature].value_counts(dropna=False).head(10))
    print()

has_pool
has_pool
NaN      925468
False     22502
True       2724
Name: count, dtype: int64

has_barbecue
has_barbecue
NaN      923745
False     20834
True       6115
Name: count, dtype: int64

has_sauna
has_sauna
NaN      926235
False     23410
True       1049
Name: count, dtype: int64

has_jacuzzi
has_jacuzzi
NaN      926036
False     23018
True       1640
Name: count, dtype: int64

has_balcony
has_balcony
NaN         444283
true        392096
false        88855
True         20692
False         4545
unselect       223
Name: count, dtype: int64

has_elevator
has_elevator
NaN      408945
True     365148
False    176601
Name: count, dtype: int64

has_warehouse
has_warehouse
True     582313
NaN      222539
False    145842
Name: count, dtype: int64

has_parking
has_parking
True     552542
NaN      222538
False    175614
Name: count, dtype: int64

has_water
has_water
NaN      921579
False     15427
True      13688
Name: count, dtype: int64

has_warm_water_provider
has_warm_water_provider
N

مقدار خالی یعنی نمی‌دانیم این امکان وجود دارد یا نه؛ آن را False نمی‌گیریم.

In [7]:
feature_values = {
    "true": True, "false": False,
    "1": True, "0": False,
    "1.0": True, "0.0": False
}

categorical_features = {
    "has_warm_water_provider": ["package", "water_heater", "powerhouse"],
    "has_heating_system": [
        "shoofaj", "heater", "duct_split", "floor_heating",
        "split", "fan_coil", "fireplace"
    ],
    "has_cooling_system": [
        "water_cooler", "air_conditioner", "split", "duct_split", "fan_coil"
    ],
    "has_restroom": ["squat", "seat", "squat_seat"]
}

for feature in all_features:
    values = housing_data[feature].astype("string").str.strip().str.lower()
    value_map = feature_values.copy()
    if feature in categorical_features:
        value_map.update({value: True for value in categorical_features[feature]})
    housing_data[feature] = values.map(value_map).astype("boolean")
    # null va unselect ro nadarad hesab nemikonim

feature_summary = pd.DataFrame({
    "with": housing_data[all_features].eq(True).sum(),
    "without": housing_data[all_features].eq(False).sum(),
    "unknown": housing_data[all_features].isna().sum()
})

feature_summary

,with,without,unknown
has_pool,2724,22502,925468
has_barbecue,6115,20834,923745
has_sauna,1049,23410,926235
has_jacuzzi,1640,23018,926036
has_balcony,412788,93400,444506
has_elevator,365148,176601,408945
has_warehouse,582313,145842,222539
has_parking,552542,175614,222538
has_water,13688,15427,921579
has_warm_water_provider,378118,0,572576


##### قیمت آگهی
در داده واقعی، `transformable_price` بولی است، نه مبلغ؛ فروش را از `price_value` می‌گیریم.
برای اجاره، `transformable_credit` رهن خام و `transformable_rent` اجاره ماهانه است.
رهن معادل = رهن + اجاره / ۰٫۰۳؛ نسبت تبدیل مطابق مثال سؤال انتخاب شده است.
فروش و اجاره و همچنین مسکونی و تجاری جدا بررسی می‌شوند. اجاره روزانه در این مقایسه نیست.

In [8]:
rent_to_credit_rate = 0.03
# har 1 million rahn = 30 hezar ejare

sale_price = pd.to_numeric(housing_data["price_value"], errors="coerce")
credit = pd.to_numeric(housing_data["transformable_credit"], errors="coerce")
rent = pd.to_numeric(housing_data["transformable_rent"], errors="coerce")

sell_mask = housing_data["cat2_slug"].isin([
    "residential-sell", "commercial-sell"
])
rent_mask = housing_data["cat2_slug"].isin([
    "residential-rent", "commercial-rent"
])

fixed_modes = ["مقطوع", "مجانی"]
valid_sell = sell_mask & housing_data["price_mode"].isin(fixed_modes)
valid_rent = (
    rent_mask
    & housing_data["credit_mode"].isin(fixed_modes)
    & housing_data["rent_mode"].isin(fixed_modes)
    & credit.ge(0)
    & rent.ge(0)
)

housing_data["price"] = np.nan
housing_data.loc[valid_sell, "price"] = sale_price[valid_sell]
housing_data.loc[valid_rent, "price"] = (
    credit[valid_rent] + rent[valid_rent] / rent_to_credit_rate
)

housing_data["price"] = housing_data["price"].replace(
    [np.inf, -np.inf], np.nan
)

print("Missing / non-comparable price:", housing_data["price"].isna().sum())
print("Non-positive price:", (housing_data["price"] <= 0).sum())

Missing / non-comparable price: 32481
Non-positive price: 1901


In [9]:
analysis_data = housing_data.dropna(subset=["price"]).copy()
analysis_data = analysis_data[analysis_data["price"] > 0].copy()

print("Number of usable ads:", len(analysis_data))

analysis_data.groupby("cat2_slug")["price"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

Number of usable ads: 916312


,count,mean,median,std,min,max
cat2_slug,,,,,,
commercial-rent,75913,3.809719e+12,5.500000e+08,2.277668e+14,34.333333,3.433333e+16
commercial-sell,31929,2.575201e+10,3.500000e+09,7.219216e+11,1.000000,1.000000e+14
residential-rent,275209,7.570731e+11,4.666667e+08,8.493659e+13,111.000000,3.433333e+16
residential-sell,533261,1.695065e+10,2.800000e+09,5.806223e+11,1.000000,1.000000e+14


##### بررسی قیمت‌های پرت
چند مقدار بسیار بزرگ میانگین را تغییر می‌دهند. مثل تمرین قبلی از صدک‌ها استفاده می‌کنیم.
در هر دسته، ۱٪ پایین و ۱٪ بالای قیمت را کنار می‌گذاریم؛ مرز برای همه امکانات آن دسته یکسان است.
نتیجه این بخش درباره همین بازه قیمت است. نتیجه داده خام را هم پایین‌تر مقایسه می‌کنیم.

In [10]:
raw_analysis_data = analysis_data.copy()

price_bounds = (
    raw_analysis_data.groupby("cat2_slug")["price"]
    .quantile([0.01, 0.99])
    .unstack()
    .rename(columns={0.01: "lower_bound", 0.99: "upper_bound"})
)

lower_bound = raw_analysis_data["cat2_slug"].map(price_bounds["lower_bound"])
upper_bound = raw_analysis_data["cat2_slug"].map(price_bounds["upper_bound"])

analysis_data = raw_analysis_data[
    raw_analysis_data["price"].between(lower_bound, upper_bound)
].copy()

print("Raw usable ads:", len(raw_analysis_data))
print("Ads after price filtering:", len(analysis_data))
price_bounds

Raw usable ads: 916312
Ads after price filtering: 899297


,lower_bound,upper_bound
cat2_slug,,
commercial-rent,4.814811e+06,3.496162e+10
commercial-sell,7.000000e+02,1.750800e+11
residential-rent,3.433333e+07,8.500000e+09
residential-sell,1.111110e+05,7.965000e+10


##### هر آگهی یک Observation
برای هر امکان، دو گروه داریم: دارای آن و فاقد آن.
مقدار نامشخص فقط از مقایسه همان امکان کنار می‌رود.

##### فرض‌ها
$\mu_{with}$ میانگین قیمت با امکان و $\mu_{without}$ میانگین قیمت بدون آن است.

لاکچری: $H_0: \mu_{with} \leq \mu_{without}$، $H_1: \mu_{with} > \mu_{without}$.

غیرلاکچری: $H_0: \mu_{with} = \mu_{without}$، $H_1: \mu_{with} \neq \mu_{without}$.

سطح معناداری: 0.05.

##### انتخاب آزمون
طبق توضیح CLT در جزوه، اگر هر دو گروه حداقل ۳۰ آگهی داشته باشند، برای میانگین‌ها از Welch t-test استفاده می‌کنیم.
برای گروه کوچک‌تر، نرمال بودن را با Shapiro بررسی می‌کنیم. اگر نرمال نبود، Mann-Whitney فقط تفاوت توزیع را بررسی می‌کند، نه میانگین را.

In [11]:
alpha = 0.05

def compare_feature(category_data, feature, alternative):
    with_ads = category_data.loc[
        category_data[feature] == True, "price"
    ]
    without_ads = category_data.loc[
        category_data[feature] == False, "price"
    ]

    row = {
        "feature": feature,
        "n_with": len(with_ads),
        "n_without": len(without_ads),
        "mean_with": with_ads.mean(),
        "mean_without": without_ads.mean(),
        "mean_difference": with_ads.mean() - without_ads.mean(),
        "change_percent": np.nan,
        "test": "Not enough known values",
        "target": "-",
        "statistic": np.nan,
        "p_value": np.nan,
        "decision": "Not tested"
    }

    if len(with_ads) and len(without_ads):
        row["change_percent"] = (
            row["mean_difference"] / row["mean_without"] * 100
        )

    if min(len(with_ads), len(without_ads)) < 3:
        return row

    large_samples = min(len(with_ads), len(without_ads)) >= 30
    normal_samples = False

    if not large_samples:
        with_shapiro = stats.shapiro(
            with_ads.sample(min(len(with_ads), 5000), random_state=42)
        )
        without_shapiro = stats.shapiro(
            without_ads.sample(min(len(without_ads), 5000), random_state=42)
        )
        normal_samples = (
            with_shapiro.pvalue >= alpha
            and without_shapiro.pvalue >= alpha
            and with_ads.std() > 0
            and without_ads.std() > 0
        )
        print(
            category_data["cat2_slug"].iloc[0], feature,
            "Shapiro p-values:",
            with_shapiro.pvalue, without_shapiro.pvalue
        )

    if large_samples or normal_samples:
        statistic, p_value = stats.ttest_ind(
            with_ads,
            without_ads,
            equal_var=False,
            alternative=alternative
        )
        row["test"] = "Welch Two-Sample t-Test"
        row["target"] = "mean"
    else:
        statistic, p_value = stats.mannwhitneyu(
            with_ads,
            without_ads,
            alternative=alternative
        )
        row["test"] = "Mann-Whitney U Test"
        row["target"] = "distribution"

    row["statistic"] = statistic
    row["p_value"] = p_value

    if np.isfinite(p_value):
        row["decision"] = (
            "Reject H0" if p_value < alpha else "Fail to reject H0"
        )
    else:
        row["decision"] = "Undefined test"

    return row

##### امکانات لاکچری

In [12]:
luxury_results = []

for category in property_categories:
    category_data = analysis_data[
        analysis_data["cat2_slug"] == category
    ]

    for feature in luxury_features:
        row = compare_feature(category_data, feature, alternative="greater")
        row["cat2_slug"] = category
        luxury_results.append(row)

luxury_results = pd.DataFrame(luxury_results)

with pd.option_context("display.max_rows", None, "display.float_format", "{:.3e}".format):
    display(luxury_results.set_index(["cat2_slug", "feature"]))

n_with  n_without  mean_with  mean_without  \
cat2_slug        feature                                                    
residential-sell has_pool        2525      17022  6.547e+09     4.651e+09   
                 has_barbecue    5506      15646  5.048e+09     4.677e+09   
                 has_sauna        983      17828  7.109e+09     4.783e+09   
                 has_jacuzzi     1503      17494  7.531e+09     4.692e+09   
residential-rent has_pool         130       4694  1.570e+09     4.967e+08   
                 has_barbecue     477       4461  1.037e+09     4.821e+08   
                 has_sauna         33       4762  1.829e+09     5.091e+08   
                 has_jacuzzi       85       4723  1.712e+09     5.013e+08   
commercial-sell  has_pool           0          0        NaN           NaN   
                 has_barbecue       0          0        NaN           NaN   
                 has_sauna          0          0        NaN           NaN   
                 has_jacuzzi        0          0        NaN           NaN   
commercial-rent  has_pool           0          0        NaN           NaN   
                 has_barbecue       0          0        NaN           NaN   
                 has_sauna          0          0        NaN           NaN   
                 has_jacuzzi        0          0        NaN           NaN   

                               mean_difference  change_percent  \
cat2_slug        feature                                         
residential-sell has_pool            1.897e+09       4.078e+01   
                 has_barbecue        3.702e+08       7.914e+00   
                 has_sauna           2.327e+09       4.865e+01   
                 has_jacuzzi         2.839e+09       6.051e+01   
residential-rent has_pool            1.073e+09       2.161e+02   
                 has_barbecue        5.550e+08       1.151e+02   
                 has_sauna           1.320e+09       2.592e+02   
                 has_jacuzzi         1.211e+09       2.415e+02   
commercial-sell  has_pool                  NaN             NaN   
                 has_barbecue              NaN             NaN   
                 has_sauna                 NaN             NaN   
                 has_jacuzzi               NaN             NaN   
commercial-rent  has_pool                  NaN             NaN   
                 has_barbecue              NaN             NaN   
                 has_sauna                 NaN             NaN   
                 has_jacuzzi               NaN             NaN   

                                                  test target  statistic  \
cat2_slug        feature                                                   
residential-sell has_pool      Welch Two-Sample t-Test   mean  1.077e+01   
                 has_barbecue  Welch Two-Sample t-Test   mean  3.474e+00   
                 has_sauna     Welch Two-Sample t-Test   mean  7.059e+00   
                 has_jacuzzi   Welch Two-Sample t-Test   mean  1.110e+01   
residential-rent has_pool      Welch Two-Sample t-Test   mean  1.168e+01   
                 has_barbecue  Welch Two-Sample t-Test   mean  1.329e+01   
                 has_sauna     Welch Two-Sample t-Test   mean  5.926e+00   
                 has_jacuzzi   Welch Two-Sample t-Test   mean  9.168e+00   
commercial-sell  has_pool      Not enough known values      -        NaN   
                 has_barbecue  Not enough known values      -        NaN   
                 has_sauna     Not enough known values      -        NaN   
                 has_jacuzzi   Not enough known values      -        NaN   
commercial-rent  has_pool      Not enough known values      -        NaN   
                 has_barbecue  Not enough known values      -        NaN   
                 has_sauna     Not enough known values      -        NaN   
                 has_jacuzzi   Not enough known values      -        NaN   

                                p_value    decision  
cat2_slug        feature        

##### امکانات غیرلاکچری

In [13]:
non_luxury_results = []

for category in property_categories:
    category_data = analysis_data[
        analysis_data["cat2_slug"] == category
    ]

    for feature in non_luxury_features:
        row = compare_feature(category_data, feature, alternative="two-sided")
        row["cat2_slug"] = category
        non_luxury_results.append(row)

non_luxury_results = pd.DataFrame(non_luxury_results)

with pd.option_context("display.max_rows", None, "display.float_format", "{:.3e}".format):
    display(non_luxury_results.set_index(["cat2_slug", "feature"]))

n_with  n_without  mean_with  \
cat2_slug        feature                                                 
residential-sell has_balcony              261060      47999  5.847e+09   
                 has_elevator             216759      82179  6.753e+09   
                 has_warehouse            354919      61188  5.890e+09   
                 has_parking              341308      74799  6.139e+09   
                 has_water                 11089      11945  4.583e+09   
                 has_warm_water_provider  223563          0  5.522e+09   
                 has_electricity           11027      12004  4.565e+09   
                 has_gas                   10934      12095  4.567e+09   
                 has_heating_system       217468          0  5.615e+09   
                 has_cooling_system       203803          0  5.766e+09   
                 has_restroom             241650          0  5.551e+09   
                 has_security_guard         5594      15928  4.097e+09   
residential-rent has_balcony              142864      43002  8.607e+08   
                 has_elevator             125499      81930  1.124e+09   
                 has_warehouse            202091      67863  8.901e+08   
                 has_parking              186008      83946  9.606e+08   
                 has_water                  2276       2941  5.810e+08   
                 has_warm_water_provider  148085          0  7.594e+08   
                 has_electricity            2253       2965  5.829e+08   
                 has_gas                    2228       2987  5.827e+08   
                 has_heating_system       142223          0  7.922e+08   
                 has_cooling_system       135090          0  8.199e+08   
                 has_restroom             157360          0  7.862e+08   
                 has_security_guard          235       4618  1.235e+09   
commercial-sell  has_balcony                   0          0        NaN   
                 has_elevator               3224       1127  1.319e+10   
                 has_warehouse              2338       2013  1.503e+10   
                 has_parking                2387       1964  1.675e+10   
                 has_water                     0          0        NaN   
                 has_warm_water_provider       0          0        NaN   
                 has_electricity               0          0        NaN   
                 has_gas                       0          0        NaN   
                 has_heating_system            0          0        NaN   
                 has_cooling_system            0          0        NaN   
                 has_restroom                  0          0        NaN   
                 has_security_guard            0          0        NaN   
commercial-rent  has_balcony                   0          0        NaN   
                 has_elevator              12052       8681  2.066e+09   
                 has_warehouse              9530      11203  2.461e+09   
                 has_parking                9325      11408  2.818e+09   
                 has_water                     0          0        NaN   
                 has_warm_water_provider       0          0        NaN   
                 has_electricity               0          0        NaN   
                 has_gas                       0          0        NaN   
                 has_heating_system            0          0        NaN   
                 has_cooling_system            0          0        NaN   
                 has_restroom                  0          0        NaN   
                 has_security_guard            0          0        NaN   

                                          mean_without  mean_difference  \
cat2_slug        feature                                                  
residential-sell has_balcony                 3.335e+09        2.512e+09   
                 has_elevator                2.797e+09        3.956e+09   
                 has_warehouse           

##### مقایسه با داده خام
از این جدول می‌بینیم کنارگذاشتن مقادیر پرت چه اثری روی نتیجه دارد.
اگر نتیجه تغییر کرد، نمی‌توان نتیجه بازه پاک‌سازی‌شده را به تمام قیمت‌های خام تعمیم داد.

In [14]:
raw_results = []

for category in property_categories:
    category_data = raw_analysis_data[
        raw_analysis_data["cat2_slug"] == category
    ]
    for feature in all_features:
        alternative = "greater" if feature in luxury_features else "two-sided"
        row = compare_feature(category_data, feature, alternative)
        row["cat2_slug"] = category
        raw_results.append(row)

raw_results = pd.DataFrame(raw_results)
filtered_results = pd.concat([luxury_results, non_luxury_results], ignore_index=True)

comparison_columns = [
    "cat2_slug", "feature", "mean_difference", "p_value", "target", "decision"
]
sensitivity_summary = raw_results[comparison_columns].merge(
    filtered_results[comparison_columns],
    on=["cat2_slug", "feature"],
    suffixes=("_raw", "_filtered")
)

with pd.option_context("display.max_rows", None, "display.float_format", "{:.3e}".format):
    display(sensitivity_summary.set_index(["cat2_slug", "feature"]))

mean_difference_raw  p_value_raw  \
cat2_slug        feature                                                     
residential-sell has_pool                          -7.240e+09    8.837e-01   
                 has_barbecue                      -7.923e+09    8.779e-01   
                 has_sauna                          1.680e+10    2.092e-01   
                 has_jacuzzi                        1.023e+10    2.371e-01   
                 has_balcony                        8.372e+09    4.709e-08   
                 has_elevator                       1.259e+10    5.117e-21   
                 has_warehouse                      8.397e+09    9.358e-12   
                 has_parking                        7.535e+09    6.437e-04   
                 has_water                          7.065e+09    4.459e-01   
                 has_warm_water_provider                  NaN          NaN   
                 has_electricity                    7.154e+09    4.425e-01   
                 has_gas                            7.305e+09    4.365e-01   
                 has_heating_system                       NaN          NaN   
                 has_cooling_system                       NaN          NaN   
                 has_restroom                             NaN          NaN   
                 has_security_guard                -1.134e+10    7.742e-02   
residential-rent has_pool                          -3.863e+12    9.360e-01   
                 has_barbecue                       1.266e+13    1.237e-01   
                 has_sauna                         -3.806e+12    9.360e-01   
                 has_jacuzzi                       -3.838e+12    9.360e-01   
                 has_balcony                       -6.227e+11    2.204e-01   
                 has_elevator                      -5.970e+11    1.823e-01   
                 has_warehouse                     -6.016e+11    9.771e-02   
                 has_parking                       -5.662e+11    2.354e-01   
                 has_water                         -4.574e+11    9.152e-01   
                 has_warm_water_provider                  NaN          NaN   
                 has_electricity                   -3.839e+11    9.285e-01   
                 has_gas                           -3.197e+11    9.403e-01   
                 has_heating_system                       NaN          NaN   
                 has_cooling_system                       NaN          NaN   
                 has_restroom                             NaN          NaN   
                 has_security_guard                -3.927e+12    1.282e-01   
commercial-sell  has_pool                                 NaN          NaN   
                 has_barbecue                             NaN          NaN   
                 has_sauna                                NaN          NaN   
                 has_jacuzzi                              NaN          NaN   
                 has_balcony                              NaN          NaN   
                 has_elevator                       8.100e+09    3.180e-01   
                 has_warehouse                      1.674e+10    5.422e-03   
                 has_parking                        1.132e+10    9.054e-02   
                 has_water                                NaN          NaN   
                 has_warm_water_provider                  NaN          NaN   
                 has_electricity                          NaN          NaN   
                 has_gas                                  NaN          NaN   
                 has_heating_system                       NaN          NaN   
                 has_cooling_system                       NaN          NaN   
                 has_restroom                             NaN          NaN   
                 has_security_guard                       NaN          NaN   
commercial-rent  has_pool                                 NaN          NaN   
                 has_barbecue                             NaN   

##### نتیجه در بازه پاک‌سازی‌شده
`p_value < 0.05` یعنی رد فرض صفر. اندازه اختلاف را در `mean_difference` و `change_percent` می‌بینیم؛ معناداری به‌تنهایی نشان نمی‌دهد افزایش بزرگ است.
`Not tested` یعنی داده کافی برای مقایسه نداریم؛ به معنی نبود تفاوت نیست.
نتیجه هر امکان جداست؛ وجود تفاوت به‌تنهایی علت تغییر قیمت را نشان نمی‌دهد.

In [15]:
for category in property_categories:
    print("\n", category, "(price-filtered data)")

    luxury_mean = luxury_results[
        (luxury_results["cat2_slug"] == category)
        & (luxury_results["target"] == "mean")
    ]
    non_luxury_mean = non_luxury_results[
        (non_luxury_results["cat2_slug"] == category)
        & (non_luxury_results["target"] == "mean")
    ]

    higher_luxury = luxury_mean.loc[
        luxury_mean["decision"] == "Reject H0", "feature"
    ].tolist()
    different_non_luxury = non_luxury_mean.loc[
        non_luxury_mean["decision"] == "Reject H0", "feature"
    ].tolist()

    print("افزایش معنادار میانگین برای امکانات لاکچری:", higher_luxury)
    print("تفاوت معنادار میانگین برای امکانات غیرلاکچری:", different_non_luxury)

    if different_non_luxury:
        print("پس برای امکانات غیرلاکچری هم تفاوت معنادار میانگین دیده می‌شود.")
    elif (non_luxury_mean["decision"] == "Fail to reject H0").any():
        print("برای امکاناتی که آزمون معتبر میانگین داشتند، شواهد کافی برای تفاوت نداریم.")
    else:
        print("برای میانگین امکانات غیرلاکچری آزمون معتبری به دست نیامد.")

    category_results = pd.concat([
        luxury_results[luxury_results["cat2_slug"] == category],
        non_luxury_results[non_luxury_results["cat2_slug"] == category]
    ])
    if (category_results["target"] == "distribution").any():
        print("رد فرض صفر در Mann-Whitney فقط درباره توزیع است؛ نتیجه میانگین نیست.")
    not_tested = category_results.loc[
        category_results["decision"].isin(["Not tested", "Undefined test"]),
        "feature"
    ].tolist()
    if not_tested:
        print("بدون نتیجه معتبر:", not_tested)


 residential-sell (price-filtered data)
افزایش معنادار میانگین برای امکانات لاکچری: ['has_pool', 'has_barbecue', 'has_sauna', 'has_jacuzzi']
تفاوت معنادار میانگین برای امکانات غیرلاکچری: ['has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'has_electricity', 'has_gas', 'has_security_guard']
پس برای امکانات غیرلاکچری هم تفاوت معنادار میانگین دیده می‌شود.
بدون نتیجه معتبر: ['has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom']

 residential-rent (price-filtered data)
افزایش معنادار میانگین برای امکانات لاکچری: ['has_pool', 'has_barbecue', 'has_sauna', 'has_jacuzzi']
تفاوت معنادار میانگین برای امکانات غیرلاکچری: ['has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'has_water', 'has_electricity', 'has_gas', 'has_security_guard']
پس برای امکانات غیرلاکچری هم تفاوت معنادار میانگین دیده می‌شود.
بدون نتیجه معتبر: ['has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom']

 commercial-sell (price-filtered data)

این نتیجه‌ها در سطح ۰٫۰۵ برای هر آزمون جدا هستند؛ نتیجه هم‌زمان همه امکانات محسوب نمی‌شوند.
امکانات دارای مقدار نامشخص وارد مقایسه نمی‌شوند؛ نتیجه فقط برای آگهی‌های با اطلاعات مشخص است.
در نمونه واقعی، امکانات لاکچری در آگهی‌های خانه و ویلا ثبت شده‌اند؛ خالی بودن ستون در آپارتمان یا ملک تجاری را به معنی نداشتن آن نمی‌گیریم.